## EE 242 Lab 3a – Frequency Domain Representation of Signals - Fourier Series

**Hanlin Ma, Amanda Zhang, Sparsh Dadhich AF07**

This lab has 2 exercises to be completed as a team. Each should be given a separate code cell in your Notebook, followed by a markdown cell with report discussion. Your notebook should start with a markdown title and overview cell, which should be followed by an import cell that has the import statements for all assignments. For this assignment, you will need to import: numpy, the wavfile package from scipy.io, and matplotlib.pyplot.  

In [ ]:
# We'll refer to this as the "import cell." Every module you import should be imported here.
%matplotlib inline
import numpy as np
import matplotlib
import scipy.signal as sig
import matplotlib.pyplot as plt
# import whatever other modules you use in this lab -- there are more that you need than we've included
from scipy.io.wavfile import read, write
from IPython.display import Audio
import IPython.display as ipd

## Summary

In this lab, we will learn how to build periodic signals from component sinusoids and how to transform signals from the time domain to the frequency domain. The concepts we’ll focus on include: implementation of the Fourier Series synthesis equation, using a discrete implementation of the Fourier Transform (DFT) with a digitized signal, and understanding the relationship between the discrete DFT index k and frequency
ω for both the original continuous signal x(t). This is a two-week lab.  You should plan on completing the first 2 assignments in the first week.

## Lab 3a turn in checklist

•	Lab 3a Jupyter notebook with code for the 2 exercises assignment in separate cells. Each assignment cell should contain markdown cells (same as lab overview cells) for the responses to lab report questions. Include your lab members’ names at the top of the notebook.

**Please submit the report as PDF**




## Assignment 1 -- Generating simple periodic signals

In the first assignment, you will develop an understanding of how some periodic signals are easier to approximate than others with a truncated Fourier Series. In this lab, we’ll work with real signals and use the synthesis equation:

$$x(t)=a_0+\sum_{k=1}^N2|a_k|cos(k\omega_0t+\angle a_k)$$

In lecture, you saw that you get ripples at transition points in approximating a square wave (**Gibbs phenomenon**). This happens for any signals with sharp edges. This assignment will involve approximating two signals (a sawtooth and a triangle wave) that have the same fundamental frequency (20Hz).

**A.** Write a function for generating a real-valued periodic time signal given the Fourier series coefficients [$a_0~a_1~···~a_N$], the sampling frequency, and the fundamental frequency. You may choose to have complex input coefficients or have separate magnitude and phase vectors for describing $a_k$.

**B.**  Define variables for the sampling frequency (8kHz) and the fundamental frequency (20Hz). Using this sampling frequency, create a time vector for a length of 200ms.

**C.**  The sawtooth signal has coefficients as follows:
$$a_0=0.5,a_k=1/(j2k\pi) $$
Using the function from part A, create three approximations of this signal with N = 2,5,20 and plot together in a 3×1 comparison.

**D.**  A triangle signal has coefficients:
$$a_0=0.5,a_k=\frac{2sin(k\pi/2)}{j(k\pi)^2}e^{-j2k\pi/2} $$
Create three approximations of this signal with N = 2,5,20 and plot together in a 3×1 comparison.


In [ ]:
# Assignment 1 - Generating Periodic Signals with Fourier Series

# Part A - Writing a periodic signal generator function
# Input: t = time in seconds, fs = sampling rate, a = coefficients above, w = fundamental frequency
def fourier_series(t, fs,a,w):
    #Step 1 : Create an empty array x(t) with number of samples as t*fs
    #Step 2 : Create a time vector t with number of samples as t*fs and each sample denoting the time
    #Step 3.1 : For every coefficient from 0 to N (You may need to find N)
    #Step 3.2 :                create a cos signal with the right parameters
    #Step 3.3 :                add this signal to the x(t) NOTE HOW THE SAMPLES ARE SAME
    #Step 4 : Return the x(t)
    N = len(a) - 1  # Number of harmonics (excluding a_0)
    num_samples = int(t * fs)  # Total number of samples
    time_vector = np.linspace(0, t, num_samples, endpoint=False)  # Time vector
    x_t = np.zeros_like(time_vector)  # Initialize x(t) with zeros

    # Add the DC component (a_0)
    x_t += a[0]

    # Add the harmonics
    for k in range(1, N + 1):
        magnitude = 2 * np.abs(a[k])  # 2|a_k|
        phase = np.angle(a[k])  # ∠a_k
        x_t += magnitude * np.cos(2 * np.pi * k * w * time_vector + phase)

    y = x_t
    return y

# Part B - Initialize the parameters
fs = 8000  # Sampling frequency (8 kHz)
w = 20  # Fundamental frequency (20 Hz)
t = 0.2  # Time duration (200 ms)

# Time vector
time_vector = np.linspace(0, t, int(t * fs), endpoint=False)


# Part C - Sawtooth Curve
# Create a vector of a values as shown above for N = 2, 5, 20
# Use the function above to find the approximations
# Plot the 3 approximations
def sawtooth_coefficients(N):
    a = [0.5]  # a_0
    for k in range(1, N + 1):
        a_k = 1 / (1j * 2 * k * np.pi)  # a_k
        a.append(a_k)
    return a

# Generate approximations
N_values = [2, 5, 20]
sawtooth_signals = []
for N in N_values:
    a = sawtooth_coefficients(N)
    x_t = fourier_series(t, fs, a, w)
    sawtooth_signals.append(x_t)

# Plot the results
plt.figure(figsize=(10, 8))
for i, N in enumerate(N_values):
    plt.subplot(3, 1, i + 1)
    plt.plot(time_vector, sawtooth_signals[i])
    plt.title(f"Sawtooth Wave Approximation (N = {N})")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

# Part D - Triangle Curve
# Create a vector of a values as shown above for N = 2, 5, 20
# Use the function above to find the approximations
# Plot the 3 approximations
def triangle_coefficients(N):
    a = [0.5]  # a_0
    for k in range(1, N + 1):
        numerator = 2 * np.sin(k * np.pi / 2)
        denominator = 1j * (k * np.pi) ** 2
        a_k = (numerator / denominator) * np.exp(-1j * 2 * k * np.pi / 2)
        a.append(a_k)
    return a

# Generate approximations
triangle_signals = []
for N in N_values:
    a = triangle_coefficients(N)
    x_t = fourier_series(t, fs, a, w)
    triangle_signals.append(x_t)

# Plot the results
plt.figure(figsize=(10, 8))
for i, N in enumerate(N_values):
    plt.subplot(3, 1, i + 1)
    plt.plot(time_vector, triangle_signals[i])
    plt.title(f"Triangle Wave Approximation (N = {N})")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()




###  Discussion

You should have noticed that the second signal converges more quickly. Discuss the two reasons for this.


The sawtooth signal decays as 1/k and the triangle signal decays at 1/k^2 which is much faster, meaning that the coefficient decrease faster for the triangle signal. Also the Gibbs phenomenon in sawtooth appears to be sharp spikes near the discontinuity but in triangle signal it appear to be sharp change in slope which also made it looks converged more quickly.

## Assignment 2 -- Synthesizing a musical note

In this assignment, you will use the same synthesis equations to try to approximate a single note from a horn, which has the frequency characteristics illustrated below. Download the file horn11short.wav from the google drive to compare your synthesized version to the original.

Figure below shows the frequency component of a note played by a horn.
![image.png](attachment:df6a5912-06a1-4792-9592-deba3281357a.png)

**A.**  Read in the horn signal, and use the sampling rate $f_s$ that you read in to create a time vector of length 100ms. Define the fundamental frequency to be $f_0$ = 335Hz. Create a signal that is a sinusoid at that frequency, and save it as a wav file.

**B.**  Create a vector (or two) to characterize $a_k$ using:

$$|a_k|:[2688,1900,316,178,78,38]$$

$$\angle a_k:[-1.73,-1.45,2.36,2.30,-2.30,1.13]$$

assuming $a_0=0$ and the first element of the vectors correspond to $a_1$. Use the function you created in part 1 to synthesize a signal, with $f_s$ and $f_0$ above, and save it as a wav file. Because the phase and magnitude are now hard coded, you may need to modify your function from above to apply here, so it is recommended that you copy and rename the function into another cell to make your debugging easier.

**C.**  Plot the 100ms section of the original file starting at 200ms with a plot of the synthesized signal in a 2×1 plot.

**D.** Play the original file, the single tone, and the 6-tone approximation in series.



In [ ]:
# Assignment 2 - Synthesizing a musical note

# Part A - Reading the signal and finding parameters
# Read the horn signal and find fs
# Define f0 (2piw) as 335 Hz
# Create a signal that is just a sinewave at f0 over 100ms using the function created in Assignment 1
# Audio file expected beside this notebook: horn11short.wav
file_name = 'horn11short.wav'
sample_rate, tr_orig = read(file_name)
tr_orig = tr_orig.astype('float64')
print(f"Original Sample Rate: {sample_rate} Hz")
print(f"tr_orig Shape: {tr_orig.shape}")# ( Total number of sampling points, 1(mono)/2(stereo))


fundamental_freq = 335 # Hz
sampling_rate = 44100 # default Hz for audio
duration = 0.1 # 100 ms

# Create a time vector for 100 ms
time_vector = np.linspace(0, duration, int(duration * sample_rate), endpoint=False)

# Create a sine wave at the fundamental frequency
sine_wave = fourier_series(duration, sampling_rate, [0,-1j], 2*np.pi*fundamental_freq)

# Save the sine wave as a WAV file
write("sine_wave.wav", sample_rate, sine_wave.astype('int16'))


# Part B - Initialize the parameters for your artificial curve.
# Create the "a" vector using the parameters given above
# Use the function created in Assignment 1 to create the signal.
magnitude = [2688, 1900, 316, 178, 78, 38]
phase = [-1.73, -1.45, 2.36, 2.30, -2.30, 1.13]

# Convert magnitude and phase to complex coefficients
a = [m * np.exp(1j * p) for m, p in zip(magnitude, phase)]

# Add a_0 = 0
a = [0] + a
def fourier_series_simple(t, fs, a, w):
    N = len(a) - 1  # Number of harmonics
    num_samples = int(t * fs)  # Total number of samples
    time_vector = np.linspace(0, t, num_samples, endpoint=False)  # Time vector
    x_t = np.zeros(num_samples)  # Initialize x(t) with zeros

    # Add the DC component (a_0)
    x_t += a[0]

    # Add the harmonics
    for k in range(1, N + 1):
        magnitude = 2 * np.abs(a[k])  # 2|a_k|
        phase = np.angle(a[k])  # ∠a_k
        x_t += magnitude * np.cos(2 * np.pi * k * w * time_vector + phase)

    return x_t

# Generate the 6-tone approximation
synthesized_signal = fourier_series_simple(duration, sample_rate, a, fundamental_freq)

# Save the synthesized signal as a WAV file
write("synthesized_signal.wav", sample_rate, synthesized_signal.astype('int16'))
# Part C - Compare the artificial signal with original signal
# Find the part of the original signal between 200ms and 300ms
# Plot this along with your artificial signal
start_time = 0.2  # 200 ms
end_time = 0.3  # 300 ms
start_sample = int(start_time * sample_rate)
end_sample = int(end_time * sample_rate)
original_signal_segment = tr_orig[start_sample:end_sample]

# Create a time vector for the segment
time_segment = np.linspace(start_time, end_time, end_sample - start_sample, endpoint=False)

# Plot the original and synthesized signals
plt.figure(figsize=(10, 8))

# Plot the original signal
plt.subplot(2, 1, 1)
plt.plot(time_segment, original_signal_segment)
plt.title("Original Signal (200 ms to 300 ms)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

# Plot the synthesized signal
plt.subplot(2, 1, 2)
plt.plot(time_vector, synthesized_signal)
plt.title("Synthesized Signal (6-Tone Approximation)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

# Part D - Make some noise
# Play the original part, sinewave and 6-tone approximations one by one in front of the TA

print("Original Signal (200 ms to 300 ms):")
ipd.display(Audio(original_signal_segment, rate=sample_rate))


print("Sine Wave at Fundamental Frequency (335 Hz):")
ipd.display(Audio(sine_wave, rate=sample_rate))


print("Synthesized Signal (6-Tone Approximation):")
ipd.display(Audio(synthesized_signal, rate=sample_rate))

###  Discussion

The approximation does not sound quite like the original signal and the plot should
look pretty different. The difference in sound is in part due to multiple factors, including the truncated approximation, imperfect estimate of the parameters, and the fact that the original signal is not perfectly periodic. Try adjusting some parameters and determine what you think is the main source of distortion.


The main source of distortion is the non-periodicity of the original signal. While truncation of the Fourier Series and imperfect parameter estimation contribute to differences, the original signal contains transient components and additional frequencies that are not purely harmonic. Since the Fourier Series assumes a perfectly periodic signal, it fails to capture these nuances, leading to noticeable discrepancies in both the waveform and sound.